# Konversi Model LightGBM + MLP → Format Snort

| ZIP Input | File | Output untuk Snort |
|---|---|---|
| `model_LightGBM.zip` | `lgbm_model_final.joblib` | `lgbm_model_final.txt` (C API) |
| `model_MLP.zip` | `mlp_model_final.h5` | `mlp_model_final.tflite` |
| Keduanya | `scaler.pkl` | `scaler_params.json` |
| Keduanya | `feature_names.pkl` | `feature_names.json` |
| Keduanya | `threshold_*.pkl` | `thresholds.json` |

---
## Sel 1 — Setup

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

!pip install lightgbm tensorflow --quiet

import os, json, zipfile, pickle, shutil
import numpy as np
import joblib
import lightgbm as lgb
import tensorflow as tf

LGBM_DIR = '/content/lgbm_files'
MLP_DIR  = '/content/mlp_files'
OUT_DIR  = '/content/snort_ready'
os.makedirs(LGBM_DIR, exist_ok=True)
os.makedirs(MLP_DIR,  exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

print('TensorFlow :', tf.__version__)
print('LightGBM   :', lgb.__version__)


---
## Sel 2 — Upload Kedua ZIP

Upload `model_LightGBM.zip` dan `model_MLP.zip` sekaligus

In [ ]:
print('Upload model_LightGBM.zip dan model_MLP.zip sekaligus...')
uploaded = files.upload()

# Ekstrak masing-masing
for fname, content in uploaded.items():
    if 'LightGBM' in fname or 'lgbm' in fname.lower():
        dest = LGBM_DIR
    elif 'MLP' in fname or 'mlp' in fname.lower():
        dest = MLP_DIR
    else:
        dest = LGBM_DIR  # default
    with zipfile.ZipFile(fname, 'r') as zf:
        zf.extractall(dest)
        print(f'\n{fname} → {dest}:')
        for f in sorted(zf.namelist()):
            path = os.path.join(dest, f)
            size = os.path.getsize(path)/1024**2
            print(f'  {f:<40} {size:.2f} MB')


---
## Sel 3 — Verifikasi File

In [ ]:
print('=== VERIFIKASI FILE LIGHTGBM ===')
lgbm_required = [
    'lgbm_model_final.joblib',
    'scaler.pkl',
    'feature_names.pkl',
    'threshold_lgbm.pkl',
]
lgbm_ok = True
for fname in lgbm_required:
    path = os.path.join(LGBM_DIR, fname)
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024**2 if exists else 0
    print(f'  [{"OK" if exists else "!!!"}] {fname:<35} {size:.2f} MB')
    if not exists: lgbm_ok = False

print(f'\n=== VERIFIKASI FILE MLP ===')
mlp_required = [
    'mlp_model_final.h5',
    'scaler.pkl',
    'feature_names.pkl',
    'threshold_mlp.pkl',
]
mlp_ok = True
for fname in mlp_required:
    path = os.path.join(MLP_DIR, fname)
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024**2 if exists else 0
    print(f'  [{"OK" if exists else "!!!"}] {fname:<35} {size:.2f} MB')
    if not exists: mlp_ok = False

print(f'\nLightGBM files: {"OK" if lgbm_ok else "ADA KURANG"}')
print(f'MLP files     : {"OK" if mlp_ok else "ADA KURANG"}')


---
## Sel 4 — Konversi LightGBM → .txt (Native C API Format)

In [ ]:
# ==============================================================================
# LightGBM tidak perlu konversi ke TFLite
# LightGBM punya C API native yang baca file .txt langsung
# lgbm_model_final.joblib → load → export ke .txt
# ==============================================================================

print('Loading lgbm_model_final.joblib...')
lgbm_model = joblib.load(os.path.join(LGBM_DIR, 'lgbm_model_final.joblib'))

print(f'Model type     : {type(lgbm_model).__name__}')
print(f'Best iteration : {lgbm_model.best_iteration_}')
print(f'N estimators   : {lgbm_model.n_estimators}')
print(f'N features     : {lgbm_model.n_features_in_}')

# Export ke format .txt (LightGBM native C API)
txt_path = os.path.join(OUT_DIR, 'lgbm_model_final.txt')
lgbm_model.booster_.save_model(txt_path)

size_joblib = os.path.getsize(os.path.join(LGBM_DIR,'lgbm_model_final.joblib'))/1024**2
size_txt    = os.path.getsize(txt_path)/1024**2
print(f'\nlgbm_model_final.joblib : {size_joblib:.2f} MB')
print(f'lgbm_model_final.txt    : {size_txt:.2f} MB')

# Verifikasi: load ulang dari .txt dan test prediksi
print('\nVerifikasi .txt dengan load ulang...')
lgbm_verify = lgb.Booster(model_file=txt_path)
dummy_38 = np.random.rand(1, 38).astype(np.float32)
pred_orig = lgbm_model.predict_proba(dummy_38)[0][1]
pred_txt  = lgbm_verify.predict(dummy_38)[0]
print(f'  P(Attack) dari .joblib : {pred_orig:.6f}')
print(f'  P(Attack) dari .txt    : {pred_txt:.6f}')
print(f'  Selisih                : {abs(pred_orig-pred_txt):.8f} (harus ~0)')
print('  LightGBM .txt OK ✓' if abs(pred_orig-pred_txt) < 1e-5 else '  WARNING: ada perbedaan!')


---
## Sel 5 — Konversi MLP → TFLite

In [ ]:
print('Loading mlp_model_final.h5...')
# compile=False untuk bypass metric incompatibility (Keras versi lama vs baru)
mlp_model = tf.keras.models.load_model(
    os.path.join(MLP_DIR, 'mlp_model_final.h5'),
    compile=False
)
mlp_model.summary()

print('\nKonversi ke TFLite...')
converter = tf.lite.TFLiteConverter.from_keras_model(mlp_model)
converter.optimizations = []  # tidak ada quantization, jaga akurasi float32
converter.target_spec.supported_types = [tf.float32]
tflite_mlp = converter.convert()

mlp_tfl_path = os.path.join(OUT_DIR, 'mlp_model_final.tflite')
with open(mlp_tfl_path, 'wb') as f:
    f.write(tflite_mlp)

size_h5  = os.path.getsize(os.path.join(MLP_DIR,'mlp_model_final.h5'))/1024
size_tfl = os.path.getsize(mlp_tfl_path)/1024
print(f'mlp_model_final.h5      : {size_h5:.1f} KB')
print(f'mlp_model_final.tflite  : {size_tfl:.1f} KB')

# Verifikasi shape
print('\nVerifikasi TFLite MLP...')
interp = tf.lite.Interpreter(model_path=mlp_tfl_path)
interp.allocate_tensors()
in_det  = interp.get_input_details()
out_det = interp.get_output_details()
print(f'  Input shape  : {in_det[0]["shape"]}  (harus [1, 38])')
print(f'  Output shape : {out_det[0]["shape"]} (harus [1, 1])')

# Verifikasi output identik dengan .h5
print('\nVerifikasi output identik .h5 vs .tflite...')
dummy_38 = np.random.rand(1, 38).astype(np.float32)
pred_h5  = mlp_model.predict(dummy_38, verbose=0)[0][0]
interp.set_tensor(in_det[0]['index'], dummy_38)
interp.invoke()
pred_tfl = interp.get_tensor(out_det[0]['index'])[0][0]
print(f'  P(Attack) dari .h5     : {pred_h5:.6f}')
print(f'  P(Attack) dari .tflite : {pred_tfl:.6f}')
print(f'  Selisih                : {abs(pred_h5-pred_tfl):.8f} (harus ~0)')
print('  MLP TFLite OK ✓' if abs(pred_h5-pred_tfl) < 1e-4 else '  WARNING: ada perbedaan!')


---
## Sel 6 — Ekspor Scaler + Feature Names (Shared)

In [ ]:
# scaler.pkl dan feature_names.pkl sama di kedua ZIP
# Pakai dari LGBM_DIR

print('=== SCALER ===')
qt = joblib.load(os.path.join(LGBM_DIR, 'scaler.pkl'))
print(f'Type            : {type(qt).__name__}')
print(f'N features      : {qt.n_features_in_}')
print(f'N quantiles     : {qt.n_quantiles_}')
print(f'Distribution    : {qt.output_distribution}')

scaler_params = {
    'type'               : 'QuantileTransformer',
    'output_distribution': qt.output_distribution,
    'n_features'         : int(qt.n_features_in_),
    'n_quantiles'        : int(qt.n_quantiles_),
    'quantiles'          : qt.quantiles_.tolist(),
    'references'         : qt.references_.tolist(),
}
scaler_path = os.path.join(OUT_DIR, 'scaler_params.json')
with open(scaler_path, 'w') as f:
    json.dump(scaler_params, f)
print(f'scaler_params.json : {os.path.getsize(scaler_path)/1024**2:.2f} MB  OK ✓')

print('\n=== FEATURE NAMES ===')
with open(os.path.join(LGBM_DIR, 'feature_names.pkl'), 'rb') as f:
    feature_names = pickle.load(f)
print(f'Jumlah fitur: {len(feature_names)}')
for i, feat in enumerate(feature_names, 1):
    print(f'  {i:2d}. {feat}')

feat_path = os.path.join(OUT_DIR, 'feature_names.json')
with open(feat_path, 'w') as f:
    json.dump({'n_features': len(feature_names), 'feature_names': feature_names}, f, indent=2)
print(f'\nfeature_names.json OK ✓')


---
## Sel 7 — Ekspor Threshold Kedua Model → thresholds.json

In [ ]:
print('Loading threshold configs...')

# LightGBM threshold
with open(os.path.join(LGBM_DIR, 'threshold_lgbm.pkl'), 'rb') as f:
    thr_lgbm = pickle.load(f)
print('LightGBM threshold config:')
for k, v in thr_lgbm.items():
    if not isinstance(v, list):
        print(f'  {k:<20} : {v}')

# MLP threshold
with open(os.path.join(MLP_DIR, 'threshold_mlp.pkl'), 'rb') as f:
    thr_mlp = pickle.load(f)
print('\nMLP threshold config:')
for k, v in thr_mlp.items():
    if not isinstance(v, list):
        print(f'  {k:<20} : {v}')

# Gabung ke satu JSON
thresholds = {
    'lgbm': {
        'threshold'   : float(thr_lgbm['threshold']),
        'n_features'  : int(thr_lgbm.get('n_features', 38)),
        'f1'          : float(thr_lgbm.get('f1', 0)),
        'fpr'         : float(thr_lgbm.get('fpr', 0)),
        'recall'      : float(thr_lgbm.get('recall', 0)),
        'model_file'  : 'lgbm_model_final.txt',
    },
    'mlp': {
        'threshold'   : float(thr_mlp['threshold']),
        'n_features'  : int(thr_mlp.get('n_features', 38)),
        'f1'          : float(thr_mlp.get('f1', 0)),
        'fpr'         : float(thr_mlp.get('fpr', 0)),
        'recall'      : float(thr_mlp.get('recall', 0)),
        'model_file'  : 'mlp_model_final.tflite',
    },
}

thr_path = os.path.join(OUT_DIR, 'thresholds_lgbm_mlp.json')
with open(thr_path, 'w') as f:
    json.dump(thresholds, f, indent=2)

print(f'\nthresholds_lgbm_mlp.json tersimpan')
print(f'  LGBM threshold : {thresholds["lgbm"]["threshold"]}')
print(f'  MLP threshold  : {thresholds["mlp"]["threshold"]}')


---
## Sel 8 — Ringkasan dan Buat ZIP

In [ ]:
print('=' * 65)
print(' RINGKASAN FILE OUTPUT UNTUK SNORT')
print('=' * 65)

expected_files = [
    # LightGBM
    ('lgbm_model_final.txt',      'LightGBM native C API format'),
    # MLP
    ('mlp_model_final.tflite',    'MLP TFLite — input [1,38] output [1,1]'),
    # Shared
    ('scaler_params.json',        'QuantileTransformer params untuk C++'),
    ('feature_names.json',        '38 nama fitur dalam urutan benar'),
    ('thresholds_lgbm_mlp.json',  'Threshold LGBM=0.68 dan MLP=0.87'),
]

all_ok = True
for fname, desc in expected_files:
    path = os.path.join(OUT_DIR, fname)
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024 if exists else 0
    unit = 'KB'
    if size > 1024:
        size /= 1024; unit = 'MB'
    status = f'OK ({size:.1f} {unit})' if exists else 'TIDAK ADA'
    print(f'  [{"OK" if exists else "!!!"}] {fname:<35} {status}')
    print(f'       {desc}')
    if not exists: all_ok = False

# Buat dua ZIP terpisah
print('\nMembuat ZIP...')

# ZIP LightGBM
zip_lgbm = '/content/snort_lgbm_ready.zip'
with zipfile.ZipFile(zip_lgbm, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['lgbm_model_final.txt', 'scaler_params.json',
                  'feature_names.json', 'thresholds_lgbm_mlp.json']:
        path = os.path.join(OUT_DIR, fname)
        if os.path.exists(path):
            zf.write(path, arcname=fname)
print(f'snort_lgbm_ready.zip : {os.path.getsize(zip_lgbm)/1024**2:.2f} MB')

# ZIP MLP
zip_mlp = '/content/snort_mlp_ready.zip'
with zipfile.ZipFile(zip_mlp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['mlp_model_final.tflite', 'scaler_params.json',
                  'feature_names.json', 'thresholds_lgbm_mlp.json']:
        path = os.path.join(OUT_DIR, fname)
        if os.path.exists(path):
            zf.write(path, arcname=fname)
print(f'snort_mlp_ready.zip  : {os.path.getsize(zip_mlp)/1024**2:.2f} MB')

print('\nDownload di Sel 9')


---
## Sel 9 — Download (Jalankan Satu per Satu)

In [ ]:
# Download LightGBM
print('Downloading snort_lgbm_ready.zip...')
files.download('/content/snort_lgbm_ready.zip')


In [ ]:
# Download MLP
print('Downloading snort_mlp_ready.zip...')
files.download('/content/snort_mlp_ready.zip')
